In [2]:
import json, random, os
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)
os.makedirs("data/raw", exist_ok=True)
print("Imports done")

Imports done


Product catalog & regions:

In [4]:
products = [
    {"product_name": "Laptop",      "category": "Electronics", "base_price": 999.99},
    {"product_name": "Smartphone",  "category": "Electronics", "base_price": 699.99},
    {"product_name": "Headphones",  "category": "Electronics", "base_price": 149.99},
    {"product_name": "Desk Chair",  "category": "Furniture",   "base_price": 249.99},
    {"product_name": "Monitor",     "category": "Electronics", "base_price": 399.99},
    {"product_name": "Keyboard",    "category": "Electronics", "base_price": 89.99},
    {"product_name": "Notebook",    "category": "Stationery",  "base_price": 4.99},
    {"product_name": "Backpack",    "category": "Accessories", "base_price": 59.99},
]
regions = ["North", "South", "East", "West", "Central"]
print("Catalog ready")

Catalog ready


Generator function with dirty data injection:

In [6]:
def random_date_2023():
    start = datetime(2023, 1, 1)
    return start + timedelta(
        days=random.randint(0, 364),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )

def generate_record(i, inject_error=False):
    product = random.choice(products)

    record = {
        "transaction_id":   f"TXN-{i:04d}",
        "customer_id":      f"CUST-{random.randint(1, 100):03d}",
        "product_name":     product["product_name"],
        "category":         product["category"],
        "price":            round(product["base_price"] * random.uniform(0.9, 1.1), 2),
        "quantity":         random.randint(1, 5),
        "discount":         round(random.choice([0, 0.05, 0.10, 0.15, 0.20]), 2),
        "region":           random.choice(regions),
        "transaction_date": random_date_2023().strftime("%Y-%m-%d %H:%M:%S"),
    }

    # ---- Inject realistic data quality problems ----
    if inject_error:
        error_type = random.choice([
            "null_customer",   # customer_id missing
            "neg_quantity",    # quantity is negative (impossible)
            "bad_discount",    # discount > 1 (should be 0.0 to 1.0 only)
            "null_price",      # price is missing
            "duplicate_txn"    # same transaction_id as an existing record
        ])

        if error_type == "null_customer":
            record["customer_id"] = None
        elif error_type == "neg_quantity":
            record["quantity"] = random.randint(-5, 0)
        elif error_type == "bad_discount":
            record["discount"] = round(random.uniform(1.1, 5.0), 2)
        elif error_type == "null_price":
            record["price"] = None
        elif error_type == "duplicate_txn":
            record["transaction_id"] = f"TXN-{random.randint(1, 200):04d}"

    return record

print("Generator function ready")

Generator function ready


Generate 800 records:

In [ ]:
records = []

# 700 clean records
for i in range(1, 701):
    records.append(generate_record(i, inject_error=False))

# 100 dirty records mixed in
for i in range(701, 801):
    records.append(generate_record(i, inject_error=True))

random.shuffle(records)  # mix dirty records throughout

print(f"Generated: {len(records)} total records")
print(f"Sample clean record:\n{json.dumps(records[0], indent=2)}")

✅ Generated: 800 total records
📋 Sample clean record:
{
  "transaction_id": "TXN-0704",
  "customer_id": "CUST-042",
  "product_name": "Notebook",
  "category": "Stationery",
  "price": null,
  "quantity": 2,
  "discount": 0.15,
  "region": "Central",
  "transaction_date": "2023-12-31 06:56:00"
}


Save to data/raw/

In [ ]:
output_path = "data/raw/sales_data.json"

with open(output_path, "w") as f:
    json.dump(records, f, indent=2)

print(f"Saved to {output_path}")
print(f"Open data/raw/sales_data.json to inspect your raw data")

✅ Saved to data/raw/sales_data.json
📁 Open data/raw/sales_data.json to inspect your raw data
